In [1]:
!nvidia-smi

Fri Jun 19 17:38:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [5]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/dodaonam/MedMAS.git"
REPO_BRANCH = "namdd19"
REPO_ROOT = Path("/content/MedMAS")

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)], check=True)
os.chdir(REPO_ROOT)

print(f"REPO_ROOT={REPO_ROOT}")
!nvidia-smi

REPO_ROOT=/content/MedMAS
Fri Jun 19 17:38:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------

In [6]:
import sys

!{sys.executable} -m pip install -q -U pip
!{sys.executable} -m pip install -q pandas python-dotenv qdrant-client fastembed langchain-core langchain-huggingface langchain-qdrant langchain-text-splitters FlagEmbedding

In [7]:
import os
import sys
from pathlib import Path

import torch

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

QDRANT_URL = os.environ.get("QDRANT_URL", "")
QDRANT_API_KEY = os.environ.get("QDRANT_API_KEY", "")
HF_TOKEN = os.environ.get("HF_TOKEN", "")

# Neu env cua runtime chua co, dien truc tiep o day.
# QDRANT_URL=
# QDRANT_API_KEY=
# HF_TOKEN=

os.environ["QDRANT_URL"] = QDRANT_URL
os.environ["QDRANT_API_KEY"] = QDRANT_API_KEY
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

INPUT_PATH = Path("/content/drive/MyDrive/vinmec_articles_structured.jsonl")
COLLECTION_NAME = os.environ.get("QDRANT_COLLECTION", "medical_docs")
FORCE_RECREATE = False
UPSERT_BATCH_SIZE = 32

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Runtime hien tai khong co CUDA.")
print("gpu:", torch.cuda.get_device_name(0))

missing_env = [name for name in ["QDRANT_URL", "QDRANT_API_KEY"] if not os.environ.get(name)]
if missing_env:
    raise RuntimeError(f"Thieu bien moi truong: {missing_env}. Dien truc tiep trong cell nay.")

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Khong tim thay file input: {INPUT_PATH}")

print(f"INPUT_PATH={INPUT_PATH}")
print(f"COLLECTION_NAME={COLLECTION_NAME}")

torch: 2.11.0+cu128
cuda available: True
gpu: Tesla T4
INPUT_PATH=/content/drive/MyDrive/vinmec_articles_structured.jsonl
COLLECTION_NAME=medical_docs


In [8]:
from qdrant_client import QdrantClient

client = QdrantClient(url=os.environ["QDRANT_URL"], api_key=os.environ.get("QDRANT_API_KEY"))
print("qdrant collections:", len(client.get_collections().collections))

qdrant collections: 1


/tmp/ipykernel_21789/2570006855.py:3: UserWarning: Api key is used with an insecure connection.
  client = QdrantClient(url=os.environ["QDRANT_URL"], api_key=os.environ.get("QDRANT_API_KEY"))


In [9]:
from ingestion.ingest import ingest_structured_jsonl

summary = ingest_structured_jsonl(
    input_path=INPUT_PATH,
    index_backend="local",
    collection_name="medical_docs",
    max_chars=5000,
    chunk_overlap=500,
    separators=("\n\n", "\n", " ", ""),
    dense_model="BAAI/bge-m3",
    sparse_model="BAAI/bge-m3",
    force_recreate=False,
    batch_size=128,
    wait=True,
)

summary

/content/MedMAS/src/ingestion/indexer_local.py:153: UserWarning: Api key is used with an insecure connection.
  client = QdrantClient(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Inference Embeddings: 100%|██████████| 2/2 [00:03<00:00,  1.80s/it]


{'input_path': '/content/drive/MyDrive/vinmec_articles_structured.jsonl',
 'index_backend': 'local',
 'collection_name': 'medical_docs',
 'documents_indexed': 390938,
 'articles_indexed': 39856,
 'dense_model': 'BAAI/bge-m3',
 'sparse_model': 'BAAI/bge-m3 lexical_weights',
 'force_recreate': False}